In [9]:
#!/usr/bin/env python3
"""
Tacotron2 Audio Generation Debugger
"""
import os
import json
import re
import torch
import logging
import numpy as np
from pathlib import Path
from datetime import datetime
import IPython.display as ipd

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('TTS-Debugger')

# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", category=FutureWarning)

# TTS imports
from TTS.tts.models.tacotron2 import Tacotron2
from TTS.tts.configs.tacotron2_config import Tacotron2Config
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.tts.utils.synthesis import synthesis

class VoiceGenerator:
    def __init__(self, config_path, checkpoint_dir, use_cpu=True):
        self.config_path = Path(config_path)
        self.checkpoint_dir = Path(checkpoint_dir)
        self.use_cpu = use_cpu
        self.device = "cpu" if use_cpu else "cuda"
        
        # Load components
        self.config = self.load_config()
        self.ap = self.init_audio_processor()
        self.tokenizer = self.init_tokenizer()
        self.model = self.load_model()
        
        logger.info("VoiceGenerator initialized successfully")
    
    def load_config(self):
        """Load and clean configuration file"""
        logger.info(f"Loading config from {self.config_path}")
        config = Tacotron2Config()
        
        # Clean JSON comments
        with open(self.config_path, 'r') as f:
            config_str = f.read()
        config_str = re.sub(r'//.*', '', config_str)
        
        # Parse cleaned JSON
        config.load_json(self.config_path)
        
        # Force CPU if requested
        if self.use_cpu:
            os.environ['CUDA_VISIBLE_DEVICES'] = ''
            
        logger.info(f"Config loaded: Model={config.model}, Sample Rate={config.audio.sample_rate}")
        return config
    
    def init_audio_processor(self):
        """Initialize audio processor"""
        logger.info("Initializing audio processor")
        ap = AudioProcessor.init_from_config(self.config)
        logger.info(f"Audio Processor: sample_rate={ap.sample_rate}, num_mels={ap.num_mels}")
        return ap
    
    def init_tokenizer(self):
        """Initialize text tokenizer"""
        logger.info("Initializing tokenizer")
        tokenizer, _ = TTSTokenizer.init_from_config(self.config)
        logger.info(f"Tokenizer: {type(tokenizer).__name__}")
        return tokenizer
    
    def find_latest_checkpoint(self):
        """Find latest checkpoint by modification time"""
        checkpoints = list(self.checkpoint_dir.glob("*.pth"))
        if not checkpoints:
            return None
            
        # Sort by modification time
        checkpoints.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        logger.info(f"Found {len(checkpoints)} checkpoints. Latest: {checkpoints[0].name}")
        return checkpoints[0]
    
    def load_model(self, checkpoint_path=None):
        """Load model with optional checkpoint"""
        logger.info("Initializing Tacotron2 model")
        
        # Find checkpoint if not provided
        if checkpoint_path is None:
            checkpoint_path = self.find_latest_checkpoint()
            if checkpoint_path is None:
                raise FileNotFoundError("No checkpoints found in directory")
        
        logger.info(f"Loading weights from {checkpoint_path}")
        
        # Initialize model
        model = Tacotron2(self.config, self.ap, self.tokenizer, speaker_manager=None)
        
        # Load weights
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        model.load_state_dict(checkpoint['model'] if 'model' in checkpoint else checkpoint)
        model.eval()
        
        logger.info(f"Model loaded: {checkpoint_path.name}")
        logger.info(f"Model device: {next(model.parameters()).device}")
        return model
    
    def generate_audio(self, text, output_path=None, debug=False):
        """Generate audio from text with optional debugging"""
        logger.info(f"Generating audio for text: '{text}'")
        
        try:
            # Run synthesis
            if debug:
                logger.info("Running synthesis in debug mode")
                
            output = synthesis(
                model=self.model,
                text=text,
                CONFIG=self.config,
                use_cuda=not self.use_cpu,
                use_griffin_lim=True,
            )
            
            # Handle different output formats
            if isinstance(output, dict):
                wav = output['wav']
                if debug:
                    logger.info(f"Output dictionary keys: {list(output.keys())}")
            else:
                wav = output
                if debug:
                    logger.info(f"Raw output type: {type(output)}")
            
            # Ensure proper shape (1D array)
            if wav.ndim > 1:
                wav = wav.flatten()
                
            # Create output filename
            if output_path is None:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                clean_text = ''.join(filter(str.isalnum, text))[:20]
                output_path = self.checkpoint_dir / f"generated_{clean_text}_{timestamp}.wav"
            
            # Save to file
            self.ap.save_wav(wav, str(output_path))
            logger.info(f"Audio saved to: {output_path}")
            
            # Return for Jupyter playback
            return wav, output_path
        
        except Exception as e:
            logger.error(f"Generation failed: {str(e)}", exc_info=True)
            return None, None

# Example usage in Jupyter
if __name__ == "__main__":
    # ===== CONFIGURATION =====
    CONFIG_PATH = "/Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/training_output/benedict_voice_finetune-June-25-2025_05+35PM-8dc80a00/config.json"
    CHECKPOINT_DIR = "/Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/training_output/benedict_voice_finetune-June-25-2025_05+35PM-8dc80a00"
    TEST_TEXT = "My dear Watson, the game is afoot and adventure awaits us both."
    DEBUG_MODE = True
    
    # Initialize generator
    generator = VoiceGenerator(
        config_path=CONFIG_PATH,
        checkpoint_dir=CHECKPOINT_DIR,
        use_cpu=True  # Set to False if using GPU
    )
    
    # Generate audio with debugging
    waveform, output_path = generator.generate_audio(
        text=TEST_TEXT,
        output_path=None,  # Auto-generate filename
        debug=DEBUG_MODE
    )
    
    # Play audio in Jupyter if successful
    if waveform is not None:
        print(f"\nGenerated audio length: {len(waveform)/generator.ap.sample_rate:.2f} seconds")
        print(f"Saved to: {output_path}")
        
        # Display audio player
        display(ipd.Audio(waveform, rate=generator.ap.sample_rate))
    else:
        print("Audio generation failed. Check logs for details.")

2025-06-26 20:19:55,825 - INFO - Loading config from /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/training_output/benedict_voice_finetune-June-25-2025_05+35PM-8dc80a00/config.json
/Users/ivkrasovskii/model-voice-generator/.venv/lib/python3.10/site-packages/coqpit/coqpit.py:898: UserWarning: Type mismatch in Tacotron2Config
Failed to deserialize field: d_vector_file (<class 'str'>) = False
Replaced it with field's default value: False
  self.deserialize(dump_dict)
2025-06-26 20:19:55,829 - INFO - Config loaded: Model=Tacotron2, Sample Rate=22050
2025-06-26 20:19:55,829 - INFO - Initializing audio processor
2025-06-26 20:19:55,832 - INFO - Audio Processor: sample_rate=22050, num_mels=80
2025-06-26 20:19:55,832 - INFO - Initializing tokenizer
2025-06-26 20:19:55,833 - INFO - Tokenizer: TTSTokenizer
2025-06-26 20:19:55,833 - INFO - Initializing Tacotron2 model
2025-06-26 20:19:55,834 - INFO - Found 4 checkpoints. Latest: checkpoint_837.pth
2025-06-26 20:19:55,8

 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:1024
 | > power:1.5
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:2.718281828459045
 | > hop_length:256
 | > win_length:1024


2025-06-26 20:19:57,508 - INFO - Output dictionary keys: ['wav', 'alignments', 'text_inputs', 'outputs']
2025-06-26 20:19:57,509 - INFO - Audio saved to: /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/training_output/benedict_voice_finetune-June-25-2025_05+35PM-8dc80a00/generated_MydearWatsonthegamei_20250626_201957.wav



Generated audio length: 5.53 seconds
Saved to: /Users/ivkrasovskii/model-voice-generator/dataset/benedict_voice_finetune/training_output/benedict_voice_finetune-June-25-2025_05+35PM-8dc80a00/generated_MydearWatsonthegamei_20250626_201957.wav
